# Track 3: seed spread for the shipping mix (`mix3`)

`sft4` scores 84/90 on the frozen hard-90 holdout, but it is **one seed**. The seed control in the
repo (`sft3_seed7`, `sft3_seed123`) was trained on the older `mix2`; the mix that actually ships has
never been trained twice, so the headline number currently has no variance estimate.

This kernel trains the identical recipe twice more (`--seed 7`, `--seed 123`; same rows, same
order of magnitude, same 2 epochs, r=16, lr 2e-4) and scores both on the same SHA-256-pinned splits.
Then it runs the paired test against `sft4` on both splits.

| step | time on T4x2 |
|---|---:|
| setup, install, 20 offline checks | 5 min |
| seed 7: train, hard-90, hard-30, upload | ~3 h |
| seed 123: train, hard-90, hard-30, upload | ~3 h |
| optional: hardened system prompt on `sft4`, hard-90 | ~1.5 h |

Order is deliberate. If the session is preempted, seed 7 is already on the Hub.
One deliberate setting: every GPU cell pins `CUDA_VISIBLE_DEVICES=0`. Kaggle's T4 x2 machine made `Trainer` wrap the 3.35B model in `DataParallel`, which keeps a full fp16 copy on each card and OOMs during the backward all-reduce. The recipe that produced `sft4` ran on a single T4, so one GPU is also the parity-correct choice.

**Quota note (read before running).** Kaggle counts a T4 x2 machine as two GPU-hours per hour, and every cell here pins one GPU anyway, so the second card only burns quota. Pick a single-GPU accelerator if Kaggle offers one (P100 / L4). The eval cell uploads hard-90 to the Hub before it starts hard-30, so a quota stop cannot take the headline number with it.


In [ ]:
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "HF_TOKEN is not attached. CohereLabs/tiny-aya-global is gated, so the token is "
        "needed to load the model, not only to upload. Add-ons -> Secrets, then Save & Run All."
    ) from exc

os.environ["HF_TOKEN"] = token
# bash cells cannot see this Python env; hand them the token through a private tmp file
# instead of the kernel output log.
env = Path("/tmp/hf_env")
env.write_text(f'export HF_TOKEN="{token}"\n', encoding="utf-8")
env.chmod(0o600)
print("HF_TOKEN set (not printed)")

!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1

## Setup: clone the repo, install, run the offline suite

The 20 checks are CPU-only and pin every frozen split by SHA-256, so anything that changed under us
fails here instead of silently inflating the score.

In [ ]:
%%bash
set -euo pipefail
source /tmp/hf_env
cd /kaggle/working
rm -rf lexsi-agentic-hinglish
git clone -q https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git
cd lexsi-agentic-hinglish
git log --oneline -1
python -m pip install -q -r requirements.txt
python scripts/verify_all.py

## Seed 7

Only the LoRA init and the data order change. `train_sft.py` writes the adapter to `runs/sft4_seed7`
and a merged fp16 copy to `runs/sft4_seed7_merged`, which is what the eval loads.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
# one T4 only: with both visible, Trainer wraps the 3.35B model in DataParallel
# (a full copy per GPU) and the backward pass OOMs at 14.3 GiB.
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
cd /kaggle/working/lexsi-agentic-hinglish
time python scripts/train_sft.py --tasks data/tasks_train_mix3.jsonl \
  --epochs 2 --seed 7 --out runs/sft4_seed7 --merge-out runs/sft4_seed7_merged \
  || echo "SEED7 TRAIN FAILED"
ls -d runs/sft4_seed7_merged 2>/dev/null || true

## Seed 7: score both frozen splits, then upload

hard-90 is the holdout the headline uses; hard-30 keeps the paired comparison with the published
`sft4` row. Upload happens before the merged checkpoint is deleted, so a preemption here costs
nothing that was already earned.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
# one T4 only: with both visible, Trainer wraps the 3.35B model in DataParallel
# (a full copy per GPU) and the backward pass OOMs at 14.3 GiB.
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f runs/sft4_seed7_merged/config.json ]; then echo "SEED7 MERGE MISSING"; exit 0; fi

time python scripts/run_baseline.py --model runs/sft4_seed7_merged   --tasks data/tasks_hard90_eval.jsonl --no-4bit --out runs/sft4_seed7_hard90   || echo "SEED7 HARD90 FAILED"

# upload the headline split before anything else: a quota stop must not cost it
python scripts/job_upload.py --run sft4_seed7_hard90   --adapter runs/sft4_seed7:sft4_seed7_adapter || echo "SEED7 UPLOAD FAILED"

time python scripts/run_baseline.py --model runs/sft4_seed7_merged   --tasks data/tasks_hard_eval.jsonl --no-4bit --out runs/sft4_seed7_hard30   || echo "SEED7 HARD30 FAILED"

python scripts/job_upload.py --run sft4_seed7_hard30 || echo "SEED7 HARD30 UPLOAD FAILED"

# free ~6.7 GB for the next seed; the adapter and both run folders stay
rm -rf runs/sft4_seed7_merged
df -h /kaggle/working | tail -1


## Seed 123

Same recipe, third initialization. With `sft4` (seed 42) this gives a three-point spread on the
shipping mix instead of a single number.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
# one T4 only: with both visible, Trainer wraps the 3.35B model in DataParallel
# (a full copy per GPU) and the backward pass OOMs at 14.3 GiB.
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
cd /kaggle/working/lexsi-agentic-hinglish
time python scripts/train_sft.py --tasks data/tasks_train_mix3.jsonl \
  --epochs 2 --seed 123 --out runs/sft4_seed123 --merge-out runs/sft4_seed123_merged \
  || echo "SEED123 TRAIN FAILED"
ls -d runs/sft4_seed123_merged 2>/dev/null || true

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f runs/sft4_seed123_merged/config.json ]; then echo "SEED123 MERGE MISSING"; exit 0; fi

time python scripts/run_baseline.py --model runs/sft4_seed123_merged   --tasks data/tasks_hard90_eval.jsonl --no-4bit --out runs/sft4_seed123_hard90   || echo "SEED123 HARD90 FAILED"

python scripts/job_upload.py --run sft4_seed123_hard90   --adapter runs/sft4_seed123:sft4_seed123_adapter || echo "SEED123 UPLOAD FAILED"

time python scripts/run_baseline.py --model runs/sft4_seed123_merged   --tasks data/tasks_hard_eval.jsonl --no-4bit --out runs/sft4_seed123_hard30   || echo "SEED123 HARD30 FAILED"

python scripts/job_upload.py --run sft4_seed123_hard30 || echo "SEED123 HARD30 UPLOAD FAILED"

rm -rf runs/sft4_seed123_merged
df -h /kaggle/working | tail -1


## Optional (last): does the hardened system prompt cost accuracy?

`safety_sft4_hardened` shows the injected-data rule buys abstention (7/8 asked) at small cost on the
safety split. The open question is whether the same extra system-prompt text slows the agent on
ordinary hard tasks. This re-merges the shipped adapter from the Hub and reruns hard-90 with
`--hardened-prompt`. Skip the cell if the session is short on time; it is the least important row.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
# one T4 only: with both visible, Trainer wraps the 3.35B model in DataParallel
# (a full copy per GPU) and the backward pass OOMs at 14.3 GiB.
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f /kaggle/working/sft4_merged/config.json ]; then
python - <<'PY'
import torch
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = "CohereLabs/tiny-aya-global"
repo = snapshot_download("primce/tiny-aya-hinglish-lora", allow_patterns=["sft4_adapter/*"],
                         local_dir="/kaggle/working/sft4_repo")
tokenizer = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, repo, subfolder="sft4_adapter")
merged = model.merge_and_unload()
merged.save_pretrained("/kaggle/working/sft4_merged", safe_serialization=True)
tokenizer.save_pretrained("/kaggle/working/sft4_merged")
print("merged -> /kaggle/working/sft4_merged")
PY
fi

if [ ! -f /kaggle/working/sft4_merged/config.json ]; then echo "SFT4 MERGE MISSING"; exit 0; fi

time python scripts/run_baseline.py --model /kaggle/working/sft4_merged --hardened-prompt \
  --tasks data/tasks_hard90_eval.jsonl --no-4bit --out runs/sft4_hard90_hardened \
  || echo "SFT4 HARDENED HARD90 FAILED"

python scripts/job_upload.py --run sft4_hard90_hardened || echo "HARDENED UPLOAD FAILED"
rm -rf /kaggle/working/sft4_merged /kaggle/working/sft4_repo
df -h /kaggle/working | tail -1

## Compare

Pulls the published `sft4` trajectories from `primce/lexsi-runs` so the paired test has both sides,
then prints the three seeds side by side and the exact McNemar result for each pair.

In [ ]:
import json
import subprocess
from pathlib import Path

from huggingface_hub import snapshot_download

ROOT = Path("/kaggle/working/lexsi-agentic-hinglish")
snapshot_download(
    "primce/lexsi-runs", repo_type="dataset",
    allow_patterns=["sft4_hard90/*", "sft4_hard30/*"],
    local_dir=str(ROOT / "runs"),
)

KEYS = ("n", "success", "calls", "steps", "ter", "rad", "lcf", "seconds")
for name in (
    "sft4_hard90", "sft4_seed7_hard90", "sft4_seed123_hard90", "sft4_hard90_hardened",
    "sft4_hard30", "sft4_seed7_hard30", "sft4_seed123_hard30",
):
    path = ROOT / "runs" / name / "summary.json"
    if not path.exists():
        print(f"{name}: MISSING")
        continue
    summary = json.loads(path.read_text(encoding="utf-8"))
    row = {key: summary.get(key) for key in KEYS}
    row["success_count"] = round((summary.get("success") or 0) * (summary.get("n") or 0))
    print(name, row)

for split, base in (("hard90", "sft4_hard90"), ("hard30", "sft4_hard30")):
    for other in (f"sft4_seed7_{split}", f"sft4_seed123_{split}"):
        if not (ROOT / "runs" / other / "trajectories.jsonl").exists():
            print(f"{other}: no trajectories, skipping test")
            continue
        print(f"\n=== {base} vs {other} ===")
        print(subprocess.run(
            ["python", "scripts/stats.py", "--a", f"runs/{base}", "--b", f"runs/{other}",
             "--label-a", base, "--label-b", other],
            cwd=ROOT, capture_output=True, text=True,
        ).stdout)

## What to paste back

The printed rows only. What the docs need:

- `sft4_seed7_hard90.success` and `sft4_seed123_hard90.success` against `sft4_hard90` (0.9333) —
  the spread, i.e. whether 84/90 is a stable result or a lucky seed.
- the McNemar pairs — if a seed is statistically indistinguishable from `sft4`, say so.
- `sft4_hard90_hardened.success` — the accuracy cost of the injection-defence system prompt.
- `seconds` for every run, so the time table in `docs/RESULTS.md` stays real.